# Healthcare Multi-Role Clinical Intelligence System
## Capstone Assignment - 8-Week Progressive Exercise

**Course**: Agentic AI Bootcamp - Staff-Level System Design

---

## Assignment Overview

In this capstone assignment, you will build a **production-grade multi-agent healthcare system** that synthesizes patient data, clinical test results, and imaging reports across multiple healthcare roles.

### What You Will Build

| Component | Description |
|-----------|-------------|
| **Lab Value Interpreter** | Analyze CBC, liver, kidney, thyroid panels |
| **Radiology Analysis Agent** | Interpret X-Ray, CT, MRI, Ultrasound reports |
| **Allergy Cross-Reactivity Checker** | Identify dangerous allergen relationships |
| **Clinical Decision Support** | Multi-agent orchestration with role-based access |
| **Medical Knowledge Graph** | Graph-based clinical reasoning |

### Dataset Overview

| Category | Files | Description |
|----------|-------|-------------|
| **Patient Records** | Asthma samples, Discharge summaries | Patient history |
| **Allergy Records** | 17 types (Food, Medication, Environment, Skin) | Allergen data |
| **Clinician/Doctor** | CBC, Liver, Kidney, Thyroid, Heart, BMI reports | Lab results |
| **Radiologist** | X-Ray, CT, MRI, Ultrasound reports | Imaging studies |

### 8-Week Structure

| Week | Module | Difficulty |
|------|--------|------------|
| 1 | Environment & Data Loading | Beginner |
| 2 | Document Processing & Entity Extraction | Beginner |
| 3 | Vector Store & Embeddings | Intermediate |
| 4 | Single Agent Design | Intermediate |
| 5 | Multi-Agent Orchestration | Advanced |
| 6 | Medical Knowledge Graph | Advanced |
| 7 | Advanced Observability | Expert |
| 8 | Production Deployment | Expert |

### Grading Rubric

- **Functionality (40%)**: All components work correctly
- **Code Quality (20%)**: Clean, documented, follows best practices
- **Observability (20%)**: Proper Langfuse tracing throughout
- **Healthcare Domain (20%)**: Correct medical logic and safety checks

---
# WEEK 1: Environment Setup & Data Loading
---

## Learning Objectives
- Configure development environment in Google Colab
- Initialize Langfuse observability from the start
- Load and explore clinical documents

## Difficulty: Beginner

**HINT**: Environment setup is foundational. Pay attention to:
- Colab vs local environment detection
- Secure API key handling
- Data directory path configuration

In [ ]:
# ============================================================================
# WEEK 1: ENVIRONMENT SETUP (PROVIDED - Run this cell as-is)
# ============================================================================

import subprocess, sys

_packages = ["openai==1.59.6", "langfuse==2.57.1", "langchain==0.3.14", "langchain-openai==0.2.14", "langchain-community==0.3.14", "langchain-core==0.3.29", "chromadb", "networkx", "pyvis", "python-docx", "openpyxl", "pdfplumber", "plotly", "seaborn", "pydantic>=2.0", "tenacity", "rich", "gradio", "python-dotenv", "numpy", "pandas", "matplotlib"]

print("Installing packages (this may take 1-2 minutes)...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + _packages,
    capture_output=True, text=True
)

if result.returncode != 0:
    for line in result.stderr.split("\n"):
        if line.strip() and "dependency resolver" not in line.lower() and "notice" not in line.lower():
            print(line)

# ---------- Verify EVERY import ----------
_verify = [
    ("openai", "openai"),
    ("langfuse", "langfuse"),
    ("langchain", "langchain"),
    ("langchain_openai", "langchain_openai"),
    ("chromadb", "chromadb"),
    ("networkx", "networkx"),
    ("pyvis", "pyvis.network"),
    ("docx", "docx"),
    ("pdfplumber", "pdfplumber"),
    ("plotly", "plotly"),
    ("seaborn", "seaborn"),
    ("pydantic", "pydantic"),
    ("tenacity", "tenacity"),
    ("rich", "rich"),
    ("gradio", "gradio"),
    ("dotenv", "dotenv"),
    ("fastapi", "fastapi"),
    ("uvicorn", "uvicorn"),
]

_failed = []
for name, imp in _verify:
    try:
        __import__(imp)
    except ImportError:
        _failed.append(name)

if _failed:
    msg = f"FATAL: These packages failed to import: {', '.join(_failed)}\n"
    msg += "Try: Runtime > Restart runtime, then re-run this cell."
    raise ImportError(msg)

print("=" * 60)
print(f"All {len(_verify)} packages installed and verified!")
print("=" * 60)

In [ ]:
# ============================================================================
# IMPORTS AND ENVIRONMENT DETECTION (PROVIDED)
# ============================================================================

import os
import sys
import json
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Optional, Any, Tuple
from dataclasses import dataclass, field
from enum import Enum
import warnings
warnings.filterwarnings('ignore')
import subprocess

IN_COLAB = 'google.colab' in sys.modules

DATASET_REPO = "https://github.com/AI-Project-Lab/IK-pwc-agenticai-datasets.git"
DATASET_PROJECT = "healthcare"

if IN_COLAB:
    print("Running in Google Colab")

    # --- API Key Configuration ---
    try:
        from google.colab import userdata
        os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
        os.environ['LANGFUSE_SECRET_KEY'] = userdata.get('LANGFUSE_SECRET_KEY')
        os.environ['LANGFUSE_PUBLIC_KEY'] = userdata.get('LANGFUSE_PUBLIC_KEY')
        _host = userdata.get('LANGFUSE_HOST') or ''
        os.environ['LANGFUSE_HOST'] = _host if _host.startswith('http') else 'https://cloud.langfuse.com'
        print("API keys loaded from Colab Secrets")
    except Exception as e:
        print(f"Colab Secrets not available: {e}")
        import getpass
        os.environ['OPENAI_API_KEY'] = getpass.getpass('Enter OpenAI API Key: ')
        os.environ['LANGFUSE_SECRET_KEY'] = getpass.getpass('Enter Langfuse Secret Key: ')
        os.environ['LANGFUSE_PUBLIC_KEY'] = getpass.getpass('Enter Langfuse Public Key: ')
        os.environ['LANGFUSE_HOST'] = 'https://cloud.langfuse.com'

    # --- Dataset Ingestion from GitHub ---
    dataset_path = "/content/datasets"
    if not os.path.exists(f"{dataset_path}/{DATASET_PROJECT}"):
        print(f"Cloning dataset from {DATASET_REPO}...")
        subprocess.run(["git", "clone", "--depth", "1", DATASET_REPO, dataset_path],
                       check=True, capture_output=True)
        print("Dataset cloned successfully!")
    else:
        print("Dataset already available.")
    DATA_DIR_BASE = f"{dataset_path}/{DATASET_PROJECT}"
else:
    print("Running in local environment")
    from dotenv import load_dotenv
    load_dotenv()
    print("API keys loaded from .env file")

    # --- Dataset Ingestion from GitHub ---
    datasets_parent = Path('.').resolve().parent.parent / 'datasets'
    if not (datasets_parent / DATASET_PROJECT).exists():
        print(f"Cloning dataset from {DATASET_REPO}...")
        subprocess.run(["git", "clone", "--depth", "1", DATASET_REPO, str(datasets_parent)],
                       check=True, capture_output=True)
        print("Dataset cloned successfully!")
    else:
        print("Dataset already available locally.")
    DATA_DIR_BASE = str(datasets_parent / DATASET_PROJECT)

# Validate API keys
required_keys = ['OPENAI_API_KEY', 'LANGFUSE_SECRET_KEY', 'LANGFUSE_PUBLIC_KEY']
missing_keys = [key for key in required_keys if not os.environ.get(key)]
if missing_keys:
    raise EnvironmentError(f"Missing required API keys: {missing_keys}")

DATA_DIR = Path(DATA_DIR_BASE)
print(f"\nAll API keys validated successfully")
print(f"Data directory: {DATA_DIR}")
print(f"Directory exists: {DATA_DIR.exists()}")
if DATA_DIR.exists():
    file_count = sum(1 for _ in DATA_DIR.rglob('*') if _.is_file())
    print(f"Total files found: {file_count}")

In [ ]:
# ============================================================================
# TODO 1.1: INITIALIZE LANGFUSE CLIENT
# ============================================================================
#
# Your task: Initialize Langfuse for observability tracking
#
# INSTRUCTIONS:
# 1. Import Langfuse and LangfuseCallbackHandler
# 2. Create a Langfuse client with proper credentials
# 3. Verify the connection with auth_check()
# 4. Create a unique SESSION_ID for this notebook run
#
# HINT: Use os.environ.get() to retrieve credentials
#
# EXPECTED OUTPUT:
# - "Langfuse connected successfully!"
# - Session ID in format: healthcare-YYYYMMDD-HHMMSS
# ============================================================================

from langfuse import Langfuse
from langfuse.callback import CallbackHandler as LangfuseCallbackHandler
from openai import OpenAI

# TODO: Initialize Langfuse client
# langfuse = Langfuse(
#     secret_key=...,
#     public_key=...,
#     host=...
# )

# YOUR CODE HERE
langfuse = None  # Replace with your implementation

# TODO: Verify connection
# langfuse.auth_check()

# TODO: Create session ID
SESSION_ID = f"healthcare-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

# Initialize OpenAI client
openai_client = OpenAI()

print(f"Session ID: {SESSION_ID}")

In [ ]:
# ============================================================================
# VALIDATION CELL 1.1: Verify Langfuse Setup
# ============================================================================

def validate_langfuse_setup():
    """Validate that Langfuse is properly configured."""
    errors = []

    if langfuse is None:
        errors.append("Langfuse client not initialized")
    else:
        try:
            langfuse.auth_check()
        except Exception as e:
            errors.append(f"Langfuse auth failed: {e}")

    if not SESSION_ID.startswith("healthcare-"):
        errors.append("SESSION_ID should start with 'healthcare-'")

    if errors:
        print("VALIDATION FAILED:")
        for e in errors:
            print(f"  - {e}")
        return False

    print("VALIDATION PASSED: Langfuse setup correct!")
    return True

validate_langfuse_setup()

In [ ]:
# ============================================================================
# TODO 1.2: CREATE TRACED HELPER FUNCTIONS
# ============================================================================
#
# Your task: Implement wrapper functions that trace all LLM operations
#
# INSTRUCTIONS:
# 1. Implement traced_embedding() - generates embeddings with Langfuse tracing
# 2. Implement traced_completion() - generates completions with Langfuse tracing
# 3. Implement get_langfuse_handler() - returns LangChain callback handler
#
# HEALTHCARE HINT: Every LLM call in healthcare MUST be traced for:
# - Audit compliance (HIPAA requirements)
# - Reproducibility of clinical decisions
# - Cost tracking per patient/query
#
# EXPECTED OUTPUT:
# Each function should log to Langfuse with proper metadata
# ============================================================================

def traced_embedding(text: str, trace_name: str = "embedding") -> List[float]:
    """
    Generate embedding with full Langfuse tracing.

    Args:
        text: The text to embed
        trace_name: Name for the Langfuse trace

    Returns:
        List of floats representing the embedding vector

    TODO:
    1. Create a Langfuse trace with session_id and metadata
    2. Create a generation span for the embedding call
    3. Call OpenAI embeddings API (model: text-embedding-3-small)
    4. End the generation with usage stats
    5. Return the embedding
    """
    # YOUR CODE HERE

    # HINT: Structure your code like this:
    # trace = langfuse.trace(name=trace_name, session_id=SESSION_ID, metadata={...})
    # generation = trace.generation(name="openai-embedding", model="text-embedding-3-small", input=...)
    # response = openai_client.embeddings.create(...)
    # embedding = response.data[0].embedding
    # generation.end(output={...}, usage={...})
    # return embedding

    pass  # Replace with your implementation


def traced_completion(prompt: str, system: str = "", trace_name: str = "completion",
                      model: str = "gpt-4o-mini", temperature: float = 0) -> str:
    """
    Generate completion with full Langfuse tracing.

    Args:
        prompt: User prompt
        system: System message
        trace_name: Name for the trace
        model: OpenAI model to use
        temperature: Sampling temperature

    Returns:
        The completion text

    TODO:
    1. Create a Langfuse trace
    2. Build messages array with system and user messages
    3. Create generation span
    4. Call OpenAI chat completions API
    5. End generation with output and token usage
    6. Return result
    """
    # YOUR CODE HERE
    pass  # Replace with your implementation


def get_langfuse_handler(trace_name: str, tags: List[str] = None) -> LangfuseCallbackHandler:
    """
    Get a LangChain callback handler for automatic tracing.

    Args:
        trace_name: Name for the trace
        tags: Optional list of tags

    Returns:
        LangfuseCallbackHandler instance

    TODO:
    Return a LangfuseCallbackHandler with proper credentials and session_id
    """
    # YOUR CODE HERE
    pass  # Replace with your implementation


print("Traced helper functions defined (implement the TODOs above)")

In [ ]:
# ============================================================================
# TODO 1.3: DISCOVER CLINICAL DATA FILES
# ============================================================================
#
# Your task: Implement a function to discover all clinical documents
#
# INSTRUCTIONS:
# 1. Recursively search the data directory
# 2. Categorize files into: patient, clinician, radiologist, training,
#    allergy_food, allergy_medication, allergy_environment, allergy_skin
# 3. Filter for supported formats: .docx, .pdf, .xlsx
#
# HEALTHCARE HINT: Proper data organization is crucial for:
# - Role-based access control (RBAC)
# - Data lineage tracking
# - Audit trails
#
# EXPECTED OUTPUT:
# Dictionary with category names as keys and lists of Path objects as values
# ============================================================================

def discover_clinical_data(data_dir: Path) -> Dict[str, List[Path]]:
    """
    Discover all clinical documents organized by category.

    Args:
        data_dir: Root directory containing clinical data

    Returns:
        Dictionary mapping category names to lists of file paths

    Categories to identify:
    - patient: Patient records and history
    - clinician: Lab reports (CBC, Liver, Kidney, etc.)
    - radiologist: Imaging reports (X-Ray, CT, MRI, Ultrasound)
    - training: Medical glossary and reference materials
    - allergy_food: Food allergy records (peanut, wheat, fish, etc.)
    - allergy_medication: Medication allergies (ibuprofen, penicillin, etc.)
    - allergy_environment: Environmental allergies (dust, pollen, etc.)
    - allergy_skin: Skin allergies and reactions
    """
    categories = {
        'patient': [],
        'clinician': [],
        'radiologist': [],
        'training': [],
        'allergy_food': [],
        'allergy_medication': [],
        'allergy_environment': [],
        'allergy_skin': []
    }

    # TODO: Check if directory exists
    if not data_dir.exists():
        print(f"Warning: Data directory not found at {data_dir}")
        return categories

    # TODO: Iterate through all files recursively
    # HINT: Use data_dir.rglob('*') to find all files
    # HINT: Check file suffix with item.suffix.lower()
    # HINT: Check path contents with 'keyword' in str(item).lower()

    # YOUR CODE HERE

    return categories


# Discover data and create trace
if langfuse:
    trace = langfuse.trace(
        name="data-discovery",
        session_id=SESSION_ID,
        metadata={"data_dir": str(DATA_DIR)}
    )

clinical_files = discover_clinical_data(DATA_DIR)

# Display results
print("\n" + "="*60)
print("HEALTHCARE DATASET DISCOVERY")
print("="*60)
total_files = 0
for category, files in clinical_files.items():
    print(f"\n{category.upper().replace('_', ' ')}: {len(files)} files")
    for f in files[:3]:
        print(f"  - {f.name}")
    if len(files) > 3:
        print(f"  ... and {len(files) - 3} more")
    total_files += len(files)

print(f"\nTotal clinical documents: {total_files}")

---
# WEEK 2: Document Processing & Entity Extraction
---

## Learning Objectives
- Parse heterogeneous clinical documents (DOCX, PDF, XLSX)
- Extract structured information from unstructured text
- Implement Named Entity Recognition for medical terms

## Difficulty: Beginner to Intermediate

**HEALTHCARE CHALLENGE**: Clinical documents have:
- Inconsistent formatting across departments
- Medical abbreviations and jargon
- Tables with lab values
- Mixed structured/unstructured content

In [ ]:
# ============================================================================
# TODO 2.1: IMPLEMENT CLINICAL DOCUMENT PROCESSOR
# ============================================================================
#
# Your task: Build a document processor that handles multiple formats
#
# INSTRUCTIONS:
# 1. Implement load_docx() - parse Word documents
# 2. Implement load_pdf() - parse PDF documents
# 3. Implement process_file() - route to correct loader
# 4. Add Langfuse tracing to process_category_with_tracing()
#
# HEALTHCARE HINT: Clinical documents often contain:
# - Tables with lab values (need special extraction)
# - Headers with patient identifiers (PHI - handle carefully!)
# - Multiple sections (History, Findings, Recommendations)
#
# EXPECTED OUTPUT:
# Dict with keys: filename, format, paragraphs, tables, full_text, table_text
# ============================================================================

from docx import Document as DocxDocument
import pdfplumber

class ClinicalDocumentProcessor:
    """
    Production-grade document processor for healthcare documents.
    Handles DOCX, PDF, and XLSX formats with Langfuse tracing.
    """

    def __init__(self):
        self.supported_formats = ['.docx', '.pdf', '.xlsx']
        self.processed_docs = []

    def load_docx(self, file_path: Path) -> Dict[str, Any]:
        """
        Load and parse a DOCX file.

        TODO:
        1. Open the document using python-docx
        2. Extract all paragraphs (filter empty ones)
        3. Extract all tables as lists of rows
        4. Return structured dictionary

        HINT for tables:
        for table in doc.tables:
            for row in table.rows:
                row_data = [cell.text.strip() for cell in row.cells]
        """
        try:
            # YOUR CODE HERE
            doc = DocxDocument(file_path)

            # TODO: Extract paragraphs
            paragraphs = []  # Extract non-empty paragraphs

            # TODO: Extract tables
            tables = []  # Extract table data

            return {
                'filename': file_path.name,
                'format': 'docx',
                'paragraphs': paragraphs,
                'tables': tables,
                'full_text': '\n'.join(paragraphs),
                'table_text': self._tables_to_text(tables)
            }
        except Exception as e:
            print(f"Error loading {file_path.name}: {e}")
            return None

    def load_pdf(self, file_path: Path) -> Dict[str, Any]:
        """
        Load and parse a PDF file.

        TODO:
        1. Open PDF with pdfplumber
        2. Extract text from each page
        3. Extract tables from each page
        4. Return structured dictionary

        HINT:
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                text = page.extract_text()
                tables = page.extract_tables()
        """
        try:
            # YOUR CODE HERE
            text_content = []
            tables = []

            # TODO: Extract content from PDF

            return {
                'filename': file_path.name,
                'format': 'pdf',
                'paragraphs': text_content,
                'tables': tables,
                'full_text': '\n'.join(text_content),
                'table_text': self._tables_to_text(tables)
            }
        except Exception as e:
            print(f"Error loading PDF {file_path.name}: {e}")
            return None

    def _tables_to_text(self, tables: List) -> str:
        """Convert tables to readable text format."""
        text_parts = []
        for i, table in enumerate(tables):
            text_parts.append(f"\nTable {i+1}:")
            for row in table:
                if isinstance(row, list):
                    text_parts.append(' | '.join(str(cell) for cell in row if cell))
        return '\n'.join(text_parts)

    def process_file(self, file_path: Path) -> Dict[str, Any]:
        """Process a single file based on its format."""
        suffix = file_path.suffix.lower()

        if suffix == '.docx':
            return self.load_docx(file_path)
        elif suffix == '.pdf':
            return self.load_pdf(file_path)
        else:
            print(f"Unsupported format: {suffix}")
            return None

    def process_category_with_tracing(self, files: List[Path], category: str) -> List[Dict]:
        """
        Process all files in a category with Langfuse tracing.

        TODO:
        1. Create a parent trace for the category
        2. Create a span for each file processed
        3. Assign doc_id to each document
        4. Update trace with final count
        """
        # YOUR CODE HERE - Add tracing
        processed = []

        for file_path in files:
            doc = self.process_file(file_path)
            if doc:
                doc['category'] = category
                doc['doc_id'] = f"{category.upper()}-{len(processed)+1:03d}"
                processed.append(doc)

        return processed


# Initialize processor
doc_processor = ClinicalDocumentProcessor()
print("ClinicalDocumentProcessor initialized")

In [ ]:
# ============================================================================
# PROCESS ALL CLINICAL DOCUMENTS
# ============================================================================

# Process all categories
all_clinical_docs = {}
for category, files in clinical_files.items():
    if files:
        all_clinical_docs[category] = doc_processor.process_category_with_tracing(files, category)
        print(f"Processed {len(all_clinical_docs[category])} {category} documents")

total_processed = sum(len(docs) for docs in all_clinical_docs.values())
print(f"\nTotal documents processed: {total_processed}")

In [ ]:
# ============================================================================
# TODO 2.2: IMPLEMENT CLINICAL ENTITY EXTRACTOR
# ============================================================================
#
# Your task: Extract medical entities from clinical text
#
# INSTRUCTIONS:
# 1. Implement extract_lab_values() - find lab test results using regex
# 2. Implement extract_status() - identify NORMAL/HIGH/LOW/CRITICAL
# 3. Implement extract_allergies() - find mentioned allergens
#
# HEALTHCARE CHALLENGE - Lab Value Interpretation:
# - Hemoglobin: Male 13.5-17.5 g/dL, Female 12.0-16.0 g/dL
# - WBC: 4,500-11,000 /uL
# - Platelet: 150,000-400,000 /uL
# - Glucose (fasting): 70-100 mg/dL
# - Creatinine: 0.7-1.3 mg/dL
#
# REGEX HINT for lab values:
# pattern = r'(?:Hemoglobin|Hgb)[:\s]+([\d.]+)\s*(g/dL)?'
# ============================================================================

import re

class ClinicalEntityExtractor:
    """
    Extract clinical entities from medical documents.
    """

    def __init__(self):
        # TODO: Define regex patterns for common lab tests
        # These patterns should match values like "Hemoglobin: 14.5 g/dL"
        self.lab_patterns = {
            'hemoglobin': r'(?:Hemoglobin|Hgb|HGB)[:\s]+([\d.]+)\s*(g/dL|g/dl)?',
            'wbc': r'(?:WBC|White Blood Cell)[:\s]+([\d,]+)\s*(/uL|/\u03bcL)?',
            # TODO: Add more patterns for:
            # - rbc (Red Blood Cell)
            # - platelet (PLT)
            # - glucose
            # - creatinine
            # - bilirubin
            # - ast (SGOT)
            # - alt (SGPT)
            # - tsh
        }

        self.status_patterns = {
            'normal': r'\b(NORMAL|Normal|WNL|Within Normal)\b',
            'high': r'\b(HIGH|High|Elevated|Increased|Above)\b',
            'low': r'\b(LOW|Low|Decreased|Below)\b',
            'critical': r'\b(CRITICAL|Critical|STAT|Urgent)\b'
        }

    def extract_lab_values(self, text: str) -> List[Dict]:
        """
        Extract lab test values from text.

        Returns list of dicts with keys: test, value, unit

        TODO:
        1. Iterate through lab_patterns
        2. Use re.findall() to find all matches
        3. Parse value and unit from matches
        4. Return list of extracted values
        """
        results = []
        # YOUR CODE HERE
        return results

    def extract_status(self, text: str) -> List[str]:
        """
        Extract status indicators from text.

        Returns list of statuses found: ['normal', 'high', etc.]
        """
        statuses = []
        # YOUR CODE HERE
        return statuses

    def extract_allergies(self, text: str) -> List[str]:
        """
        Extract potential allergens mentioned in text.

        Common allergens to detect:
        - Food: peanut, wheat, milk, egg, soy, fish, shellfish, gluten
        - Medication: penicillin, sulfa, aspirin, ibuprofen, nsaid
        - Environmental: dust, pollen, mold, pet dander, latex, bee
        """
        text_lower = text.lower()
        allergens = [
            'peanut', 'wheat', 'milk', 'egg', 'soy', 'fish', 'shellfish', 'gluten',
            'penicillin', 'sulfa', 'aspirin', 'ibuprofen', 'nsaid', 'paracetamol',
            'dust', 'pollen', 'mold', 'pet dander', 'latex', 'bee'
        ]
        # YOUR CODE HERE - return allergens found in text
        return []

    def process_document_with_tracing(self, doc: Dict) -> Dict:
        """
        Extract all entities with Langfuse tracing.

        TODO:
        1. Create a trace for this extraction
        2. Combine full_text and table_text
        3. Extract lab values, statuses, and allergies
        4. Update trace with results
        5. Return extraction results
        """
        text = doc.get('full_text', '') + '\n' + doc.get('table_text', '')

        result = {
            'doc_id': doc.get('doc_id'),
            'category': doc.get('category'),
            'lab_values': self.extract_lab_values(text),
            'statuses': self.extract_status(text),
            'allergies': self.extract_allergies(text)
        }

        # TODO: Add Langfuse tracing here

        return result


# Initialize extractor
entity_extractor = ClinicalEntityExtractor()
print("ClinicalEntityExtractor initialized")

In [ ]:
# ============================================================================
# VALIDATION CELL 2.1: Test Entity Extraction
# ============================================================================

def validate_entity_extraction():
    """Test the entity extractor with sample clinical text."""

    test_text = """
    CBC Report:
    Hemoglobin: 10.2 g/dL (LOW)
    WBC: 12,500 /uL (HIGH)
    Platelet: 180,000 /uL (NORMAL)

    Patient has documented peanut allergy.
    Avoid penicillin due to previous reaction.
    """

    lab_values = entity_extractor.extract_lab_values(test_text)
    statuses = entity_extractor.extract_status(test_text)
    allergies = entity_extractor.extract_allergies(test_text)

    print("Entity Extraction Test Results:")
    print(f"  Lab values found: {len(lab_values)}")
    for lv in lab_values:
        print(f"    - {lv}")
    print(f"  Statuses found: {statuses}")
    print(f"  Allergies found: {allergies}")

    # Validation checks
    passed = True
    if len(lab_values) < 1:
        print("  WARNING: Should find at least 1 lab value")
        passed = False
    if 'low' not in statuses and 'high' not in statuses:
        print("  WARNING: Should find LOW or HIGH status")
        passed = False
    if 'peanut' not in allergies:
        print("  WARNING: Should find peanut allergy")
        passed = False

    if passed:
        print("\nVALIDATION PASSED!")
    else:
        print("\nVALIDATION NEEDS WORK - implement the TODO sections above")

validate_entity_extraction()